# Chapter 10 — The Backward Pass, Derived

Every hand-derived gradient in Chapter 10, checked against autograd, plus every worked number and
exercise answer. Each cell **asserts** what it checks, so `python tests/run_notebooks.py 10` fails
if the chapter and the code disagree. Float64 throughout unless a cell says otherwise, as the
chapter recommends.

In [ ]:
import math, sys, os
sys.path.insert(0, os.path.abspath(".."))      # run from notebooks/
sys.path.insert(0, os.path.abspath("."))       # or from the repo root
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(0)
torch.set_default_dtype(torch.float64)

## 10.1 — matmul: the shapes force dA = dC Bᵀ and dB = Aᵀ dC

In [ ]:
A = torch.randn(4, 3, requires_grad=True); Bm = torch.randn(3, 5, requires_grad=True)
Cm = A @ Bm; g = torch.randn_like(Cm)
Cm.backward(g)
assert A.grad.shape == A.shape == (4, 3) and Bm.grad.shape == (3, 5)
assert torch.allclose(A.grad, g @ Bm.detach().T) and torch.allclose(Bm.grad, A.detach().T @ g)

## 10.2 — the softmax Jacobian at x = [1.0, 2.0, 0.5]

In [ ]:
x = torch.tensor([1.0, 2.0, 0.5])
s = x.softmax(0)
J_auto = torch.autograd.functional.jacobian(lambda t: t.softmax(0), x)
J_formula = torch.diag(s) - torch.outer(s, s)
print("s =", [round(v, 4) for v in s.tolist()])
print(J_formula.numpy().round(4))
assert [round(v, 4) for v in s.tolist()] == [0.2312, 0.6285, 0.1402]
assert torch.allclose(J_auto, J_formula)
assert J_formula.numpy().round(4).tolist() == [[0.1778, -0.1453, -0.0324],
                                               [-0.1453, 0.2335, -0.0881],
                                               [-0.0324, -0.0881, 0.1206]]
assert torch.allclose(J_formula, J_formula.T) and torch.allclose(J_formula.sum(1), torch.zeros(3))

## 10.2 — the vector-Jacobian product, which is what actually runs

In [ ]:
def softmax_backward(s, g):
    return s * (g - (g * s).sum(-1, keepdim=True))
x = torch.tensor([1.0, 2.0, 0.5], requires_grad=True)
s = x.softmax(0); g = torch.tensor([0.3, -1.2, 0.7])
s.backward(g)
assert torch.allclose(x.grad, softmax_backward(s.detach(), g), atol=1e-6)
assert torch.allclose(x.grad, J_formula.T @ g)                     # = J^T g, never forming J

## 10.3 — cross-entropy + softmax collapses to p − y (logits [2.0, 1.0, 0.1], class 0)

In [ ]:
z = torch.tensor([[2.0, 1.0, 0.1]], requires_grad=True)
F.cross_entropy(z, torch.tensor([0])).backward()
p = z.detach().softmax(-1)[0]
y = torch.tensor([1.0, 0, 0])
print("p", [round(v, 4) for v in p.tolist()], " p - y", [round(v, 4) for v in (p - y).tolist()])
assert [round(v, 4) for v in p.tolist()] == [0.659, 0.2424, 0.0986]
assert [round(v, 4) for v in (p - y).tolist()] == [-0.341, 0.2424, 0.0986]
assert torch.allclose(z.grad[0], p - y)

## 10.4 — RMSNorm's correction term, at x = [3, 1, 4, 8], g = [1, 0, 0, 0]

In [ ]:
def rmsnorm_backward(x, g, gamma, eps=0.0):
    d = x.shape[-1]
    r = x.pow(2).mean(-1, keepdim=True).add(eps).sqrt()
    gg = g * gamma
    return (gg - x * (gg * x).sum(-1, keepdim=True) / (d * r ** 2)) / r
x = torch.tensor([3., 1., 4., 8.], requires_grad=True)
gamma = torch.ones(4); g = torch.tensor([1., 0., 0., 0.])
(x / x.pow(2).mean().sqrt() * gamma).backward(g)
print("autograd", [round(v, 6) for v in x.grad.tolist()])
assert round(x.pow(2).mean().sqrt().item(), 4) == 4.7434
assert [round(v, 6) for v in x.grad.tolist()] == [0.189737, -0.007027, -0.028109, -0.056218]
assert torch.allclose(x.grad, rmsnorm_backward(x.detach(), g, gamma), atol=1e-12)
print("dot(grad, x) =", torch.dot(x.grad, x.detach()).item())
assert abs(torch.dot(x.grad, x.detach())) < 1e-12                   # orthogonal to x
# and the same holds for the float32 module in tfs/model.py, with a learned gamma
from tfs.model import RMSNorm
xr = torch.randn(5, 16, dtype=torch.float32, requires_grad=True)
nrm = RMSNorm(16, eps=0.0).float()
with torch.no_grad(): nrm.g.uniform_(0.5, 1.5)
gr = torch.randn(5, 16, dtype=torch.float32)
nrm(xr).backward(gr)
assert torch.allclose(xr.grad, rmsnorm_backward(xr.detach(), gr, nrm.g.detach()), atol=1e-5)

## 10.5 — attention backward: dQ, dK, dV against autograd (T = 4, d = 3)

In [ ]:
def attention_backward(q, k, v, P, dO):
    dk_ = q.shape[-1]
    dV = P.transpose(-2, -1) @ dO
    dP = dO @ v.transpose(-2, -1)
    dS = P * (dP - (dP * P).sum(-1, keepdim=True))
    dQ = dS @ k / dk_ ** 0.5
    dK = dS.transpose(-2, -1) @ q / dk_ ** 0.5
    return dQ, dK, dV
torch.manual_seed(0)
q, k, v = (torch.randn(4, 3, requires_grad=True) for _ in range(3))
P = (q @ k.T / 3 ** 0.5).softmax(-1)
O = P @ v
dO = torch.randn_like(O)
O.backward(dO)
mine = attention_backward(q.detach(), k.detach(), v.detach(), P.detach(), dO)
for auto, man in zip((q.grad, k.grad, v.grad), mine):
    assert torch.allclose(auto, man, atol=1e-5)
print("dQ, dK, dV all match autograd")

## 10.6 — the residual connection over depth: 0.9ᴸ, and 1 + f′

In [ ]:
for L, want, weaker in ((20, 0.1216, 8), (50, 0.0052, 190), (100, 0.0000266, 37_600)):
    print(f"{L:3d} layers: 0.9^{L} = {0.9 ** L:.7f}  ({1 / 0.9 ** L:,.0f}x weaker)")
    assert round(0.9 ** L, 7 if L == 100 else 4) == want
    assert abs(1 / 0.9 ** L - weaker) / weaker < 0.03            # 8x, ~190x, ~37,600x
x = torch.tensor(2.0, requires_grad=True)
(x + 0.3 * x ** 2).backward()
assert abs(x.grad.item() - 2.2) < 1e-12                           # 1 + f'(2) = 1 + 1.2

## Exercise 10.1 — the chapter's `check` harness on every layer, including a linear layer

In [ ]:
def check(fn_forward, manual_backward, *inputs):
    ins = [t.clone().requires_grad_(True) for t in inputs]
    out = fn_forward(*ins)
    g = torch.randn_like(out)
    out.backward(g)
    mine = manual_backward(*[t.detach() for t in inputs], g)
    for auto, man in zip([t.grad for t in ins], mine):
        assert torch.allclose(auto, man, atol=1e-5), (auto, man)
    return "ok"
torch.manual_seed(0)
X, W, b = torch.randn(6, 5), torch.randn(5, 4), torch.randn(4)
assert check(lambda x, w, b: x @ w + b, lambda x, w, b, g: (g @ w.T, x.T @ g, g.sum(0)), X, W, b) == "ok"
assert check(lambda x: x.softmax(-1), lambda x, g: (softmax_backward(x.softmax(-1), g),), torch.randn(3, 7)) == "ok"
ga = torch.rand(8) + 0.5
assert check(lambda x: x / x.pow(2).mean(-1, keepdim=True).sqrt() * ga,
             lambda x, g: (rmsnorm_backward(x, g, ga),), torch.randn(3, 8)) == "ok"
def attn_fwd(q, k, v):
    return (q @ k.transpose(-2, -1) / q.shape[-1] ** 0.5).softmax(-1) @ v
def attn_bwd(q, k, v, g):
    P = (q @ k.transpose(-2, -1) / q.shape[-1] ** 0.5).softmax(-1)
    return attention_backward(q, k, v, P, g)
assert check(attn_fwd, attn_bwd, *(torch.randn(2, 6, 4) for _ in range(3))) == "ok"
print("linear, softmax, RMSNorm, attention: ok")

## Exercise 10.2 — gradient checking with central differences

**Minor correction.** The chapter says forward differences give ~1e-6 error and central
differences ~1e-12 at ε = 1e-6. The truncation errors are O(ε) and O(ε²), as stated, but in
float64 the central difference is limited by rounding, ~machine-ε/ε ≈ 1e-10: measured below it
lands at ~1e-11 to 1e-10, not 1e-12. Still four orders better than the forward difference, which
is the point. In float32 the same check is useless, as the chapter warns.

In [ ]:
def grad_check(f, x, eps=1e-6, dtype=torch.float64):
    x = x.to(dtype)
    num = torch.zeros_like(x)
    for i in range(x.numel()):
        xp, xm = x.clone(), x.clone()
        xp.view(-1)[i] += eps; xm.view(-1)[i] -= eps
        num.view(-1)[i] = (f(xp) - f(xm)) / (2 * eps)
    xa = x.clone().requires_grad_(True); f(xa).backward()
    return ((num - xa.grad).norm() / (num.norm() + xa.grad.norm())).item()
torch.manual_seed(0)
loss = lambda t: (t.softmax(-1) * torch.arange(6.0, dtype=t.dtype)).sum() + t.pow(2).mean().sqrt()
rel64 = grad_check(loss, torch.randn(6))
rel32 = grad_check(loss, torch.randn(6), eps=1e-3, dtype=torch.float32)
print(f"relative error: float64 {rel64:.1e}   float32 {rel32:.1e}")
assert rel64 < 1e-7 and rel32 > 1e-5
x1 = torch.tensor(1.0)
fwd = ((torch.sin(x1 + 1e-6) - torch.sin(x1)) / 1e-6 - math.cos(1)).abs().item()
cen = ((torch.sin(x1 + 1e-6) - torch.sin(x1 - 1e-6)) / 2e-6 - math.cos(1)).abs().item()
print(f"d sin/dx at 1: forward-difference error {fwd:.1e}, central {cen:.1e}")
assert 1e-7 < fwd < 1e-5 and cen < 1e-9 and fwd / cen > 1e3

## Exercise 10.3 — a 40-layer tanh stack, with and without the residual

Gradient of the loss with respect to each layer's input (the signal backprop carries).

**DISCREPANCY (init-dependent figure).** The chapter expects "4–6 orders of magnitude between
layer 40 and layer 1" without residuals. The size depends entirely on the weight scale, which the
exercise does not fix: with PyTorch's default `nn.Linear` init (std ≈ 1/√(3d)) the decay is
~10 orders of magnitude; with std = 1/√d it is barely one. The qualitative result holds either way:
geometric decay without the residual, and with it the gradient does not decay at all (it grows a
little toward layer 1).

In [ ]:
def profile(residual, std=None, d=64, L=40):
    torch.manual_seed(0)
    Ws = [nn.Linear(d, d, bias=False) for _ in range(L)]
    if std is not None:
        for W in Ws: nn.init.normal_(W.weight, 0, std)
    h = torch.randn(32, d, requires_grad=True); hs = []
    for W in Ws:
        hs.append(h); h.retain_grad()
        h = h + torch.tanh(W(h)) if residual else torch.tanh(W(h))
    (h @ torch.randn(d)).sum().backward()
    return [t.grad.norm().item() for t in hs]                      # layer 1 ... layer 40
for name, std in (("default init", None), ("std 1/sqrt(d)", 1 / 8)):
    plain, res = profile(False, std), profile(True, std)
    orders = math.log10(plain[-1] / plain[0])
    print(f"{name:14}: no residual, layer40/layer1 = 10^{orders:.1f};  residual: {res[0] / res[-1]:.1f}x")
    assert res[0] >= res[-1]                                        # residual: no vanishing
    if std is None:
        assert orders > 6                                           # more than the chapter's 4-6
    else:
        assert orders < 4                                           # less than the chapter's 4-6

## Exercise 10.4 — break cross-entropy on purpose

`a` (logits) and `c` (`log(softmax)` then `nll_loss`) agree; `b` softmaxes twice. At ×50 the small
probabilities underflow to exactly 0, so `c` hits log(0) = −inf, while the fused `a` stays finite.

In [ ]:
torch.set_default_dtype(torch.float32)              # the failure is a float32 one
torch.manual_seed(0)
logits = torch.randn(4, 10, requires_grad=True)
target = torch.randint(0, 10, (4,))
def loss_and_grad(fn):
    if logits.grad is not None: logits.grad = None
    out = fn(); out.backward()
    return out.item(), logits.grad.clone()
a, ga = loss_and_grad(lambda: F.cross_entropy(logits, target))
b, gb = loss_and_grad(lambda: F.cross_entropy(logits.softmax(-1), target))
c, gc = loss_and_grad(lambda: F.nll_loss(logits.softmax(-1).log(), target))
print(f"a {a:.4f}  b {b:.4f}  c {c:.4f};  |grad a - grad b| = {(ga - gb).norm():.3f}")
assert abs(a - c) < 1e-5 and torch.allclose(ga, gc, atol=1e-6)
assert abs(a - b) > 0.1 and (ga - gb).norm() > 0.1
assert b > math.log(10) - 0.2 * math.log(10)                       # double softmax: nearly uniform
big = (logits.detach() * 50).requires_grad_(True)
t_low = big.argmin(-1)                                              # targets whose probability underflows
a50 = F.cross_entropy(big, t_low)
c50 = F.nll_loss(big.softmax(-1).log(), t_low)
print(f"x50: a = {a50.item():.1f}   c = {c50.item()}   zeros in softmax: {(big.softmax(-1) == 0).sum().item()}")
assert (big.softmax(-1) == 0).any() and math.isinf(c50.item()) and math.isfinite(a50.item())